# **Strava Fraud Detection - Model Monitoring**

## *Model Monitoring Overview:*
This notebook demonstrates how to set up comprehensive model monitoring for our Strava fraud detection models using Snowflake's ML Observability features. We'll monitor model performance, detect data drift, and track prediction quality over time.

## *Why Model Monitoring Matters:*
- **Performance Tracking**: Monitor accuracy, precision, recall, and F1-score over time
- **Data Drift Detection**: Identify when input data patterns change
- **Model Degradation**: Catch when models start performing poorly
- **Business Impact**: Ensure fraud detection remains effective as user behavior evolves

## *What We'll Demonstrate:*
- **Inference Pipeline**: Create stored procedures for model inference
- **Model Monitors**: Set up monitoring for both model versions
- **Drift Detection**: Monitor feature drift and prediction drift
- **Performance Metrics**: Track model accuracy and business metrics
- **Alerting**: Set up alerts for model performance degradation

## *Prerequisites:*
- Completed notebook 03 (Strava ML Demo) with trained models
- Models registered in Model Registry: `fraud_detection_rf` v1 and v2
- Training and test data available in feature store

## 1. Setup and Connection


In [106]:
# 📦 Install Packages, Import Libraries & Setup Connection
import sys
import subprocess

def install_package(package):
    """Install a package using pip"""
    try:
        # Handle special import name mappings
        import_name = package.split('==')[0].replace('-', '_')
        if package.startswith('snowflake-ml-python'):
            import_name = 'snowflake.ml'
        elif package.startswith('snowflake-snowpark-python'):
            import_name = 'snowflake.snowpark'
        elif package.startswith('snowflake-telemetry-python'):
            import_name = 'snowflake.telemetry'
        
        __import__(import_name)
        print(f"✅ {package} already installed")
    except ImportError:
        print(f"📦 Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        print(f"✅ {package} installed successfully")

# List of required packages for model monitoring
required_packages = [
    "snowflake-snowpark-python",
    "snowflake-ml-python",
    "snowflake-telemetry-python",
    "pandas",
    "numpy", 
    "matplotlib",
    "cryptography"
]

print("🔍 Checking and installing required packages...")
for package in required_packages:
    install_package(package)

print("✅ All packages ready!")
print("="*60)

# Import required packages for Snowflake ML monitoring
import snowflake.snowpark as snowpark
from snowflake.ml.registry import Registry
from snowflake.snowpark.types import StringType, IntegerType, FloatType, BooleanType
from snowflake.snowpark.functions import col, current_timestamp
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

print("📚 Libraries imported successfully!")

# Suppress version mismatch warnings for telemetry package
import warnings
import logging
warnings.filterwarnings("ignore", category=UserWarning, module="snowflake.snowpark.session")
warnings.filterwarnings("ignore", message=".*snowflake-telemetry-python.*")
logging.getLogger("snowflake.snowpark.session").setLevel(logging.ERROR)
print("="*60)


🔍 Checking and installing required packages...
✅ snowflake-snowpark-python already installed
✅ snowflake-ml-python already installed
✅ snowflake-telemetry-python already installed
✅ pandas already installed
✅ numpy already installed
✅ matplotlib already installed
✅ cryptography already installed
✅ All packages ready!
📚 Libraries imported successfully!


In [107]:
# 🔗 Snowflake Connection Setup
print("🔗 Establishing Snowflake connection...")

# Get Snowflake session (works in both Snowflake Notebooks and locally in Cursor)
try:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
    print("✅ Running in Snowflake Notebooks")
except:
    from snowflake_connection import get_local_session
    session = get_local_session()
    print("✅ Running locally in Cursor")

# Verify connection
print("✅ Connected to Snowflake")
print(f"🏢 Account: {session.get_current_account()}")
print(f"👤 User: {session.get_current_user()}")
print(f"🏗️ Warehouse: {session.get_current_warehouse()}")
print(f"🗃️ Database: {session.get_current_database()}")
print(f"📁 Schema: {session.get_current_schema()}")

# Ensure we're in the right context for this demo
if session.get_current_database() != "STRAVA_DEMO_SAMPLE":
    session.use_database("STRAVA_DEMO_SAMPLE")
    print("🔄 Switched to STRAVA_DEMO_SAMPLE database")

if session.get_current_schema() != "STRAVA_MODEL_REGISTRY":
    session.use_schema("STRAVA_MODEL_REGISTRY")
    print("🔄 Switched to STRAVA_MODEL_REGISTRY schema")

print(f"🎯 Ready to work with {session.get_current_database()}.{session.get_current_schema()}")
print("="*60)


🔗 Establishing Snowflake connection...
✅ Running in Snowflake Notebooks
✅ Connected to Snowflake
🏢 Account: "SFSENORTHAMERICA-ARYEUNG_AWS1"
👤 User: "aryeung"
🏗️ Warehouse: "STRAVA_DEMO_WH"
🗃️ Database: "STRAVA_DEMO_SAMPLE"
📁 Schema: "STRAVA_MODEL_REGISTRY"
🔄 Switched to STRAVA_DEMO_SAMPLE database
🔄 Switched to STRAVA_MODEL_REGISTRY schema
🎯 Ready to work with "STRAVA_DEMO_SAMPLE"."STRAVA_MODEL_REGISTRY"


## 2. Model Registry Setup

Connect to our existing Model Registry and verify our trained models are available for monitoring.


In [108]:
# Connect to Model Registry
print("🔧 Connecting to Model Registry...")

mr = Registry(
    session=session,
    database_name="STRAVA_DEMO_SAMPLE",
    schema_name="STRAVA_MODEL_REGISTRY"
)

print("✅ Model Registry connected!")

# Verify fraud detection model exists
print("\n📦 Checking for fraud detection model...")
try:
    test_model = mr.get_model("fraud_detection_rf")
    print(f"✅ Found model: {test_model.name}")
except Exception as e:
    print(f"❌ Could not find fraud_detection_rf model: {e}")
    print("💡 Make sure you've run notebook 03 to train and register models")
    raise


🔧 Connecting to Model Registry...
✅ Model Registry connected!

📦 Checking for fraud detection model...
✅ Found model: FRAUD_DETECTION_RF


In [109]:
# Get our fraud detection model and its versions
print("🔍 Checking fraud detection model versions...")

try:
    # Get the model
    fraud_model = mr.get_model("fraud_detection_rf")
    print(f"✅ Found model: {fraud_model.name}")
    
    # List all versions
    versions = fraud_model.versions()
    print(f"\n📋 Available model versions:")
    for version in versions:
        print(f"  - Version: {version.version_name}")
        if version.comment:
            print(f"    Comment: {version.comment}")
        print()
    
    # Store version references for monitoring
    model_v1 = fraud_model.version("v1")
    model_v2 = fraud_model.version("v2")
    
    print("✅ Model versions loaded successfully!")
    print(f"   - V1: {model_v1.version_name}")
    print(f"   - V2: {model_v2.version_name}")
    
except Exception as e:
    print(f"❌ Error accessing fraud detection model: {e}")
    print("💡 Make sure you've run notebook 03 to train and register the models")
    raise


🔍 Checking fraud detection model versions...
✅ Found model: FRAUD_DETECTION_RF

📋 Available model versions:
  - Version: V1
    Comment: Random Forest for fraud detection

  - Version: V2
    Comment: Improved Random Forest v2

✅ Model versions loaded successfully!
   - V1: V1
   - V2: V2


## 3. Prepare Training and Test Data for Monitoring

We need to save our training and test datasets with predictions to set up monitoring. This creates the baseline and source data for our model monitors.


In [110]:
# Get training and test data from feature store
print("📊 Preparing training and test data for monitoring...")

# First, let's check what tables exist in the feature store
print("\n🔍 Checking available tables in STRAVA_FEATURE_STORE...")
try:
    available_tables = session.sql("""
        SELECT TABLE_NAME 
        FROM STRAVA_DEMO_SAMPLE.INFORMATION_SCHEMA.TABLES 
        WHERE TABLE_SCHEMA = 'STRAVA_FEATURE_STORE'
    """).collect()
    
    if available_tables:
        print("✅ Found tables in STRAVA_FEATURE_STORE:")
        for table in available_tables:
            print(f"   - {table['TABLE_NAME']}")
    else:
        print("⚠️ No tables found in STRAVA_FEATURE_STORE schema")
        print("💡 Please run notebook 03 first to create the feature store")
        raise Exception("Feature store is empty")
    
except Exception as e:
    print(f"❌ Error checking schema: {e}")
    print("💡 Make sure you've run notebook 03 to create the feature store")
    raise

# Now try to access the athlete_features table
print("\n📊 Loading feature data...")
try:
    # Try with fully qualified name first
    feature_data = session.table("STRAVA_DEMO_SAMPLE.STRAVA_FEATURE_STORE.ATHLETE_FEATURES")
    row_count = feature_data.count()
    print(f"✅ Loaded feature data: {row_count} rows")
    
    if row_count == 0:
        print("⚠️ Warning: ATHLETE_FEATURES table is empty!")
        print("💡 Please run notebook 03 to populate the feature data")
        raise Exception("ATHLETE_FEATURES table is empty")
    
    # Split data into training (70%) and test (30%) sets
    training_data = feature_data.filter(col("ATHLETE_ID") % 10 < 7)  # 70% for training
    test_data = feature_data.filter(col("ATHLETE_ID") % 10 >= 7)     # 30% for testing
    
    print(f"✅ Training data: {training_data.count()} rows")
    print(f"✅ Test data: {test_data.count()} rows")
    
    # Show sample of the data
    print("\n📋 Sample training data:")
    training_data.limit(3).show()
    
except Exception as e:
    print(f"❌ Error loading feature data: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Check that notebook 03 completed successfully")
    print("   2. Verify ATHLETE_FEATURES table exists and has data")
    print("   3. Check you have permissions to access STRAVA_FEATURE_STORE schema")
    raise


📊 Preparing training and test data for monitoring...

🔍 Checking available tables in STRAVA_FEATURE_STORE...
✅ Found tables in STRAVA_FEATURE_STORE:
   - FRAUD_DETECTION_FEATURES_VIEW$1.0
   - ATHLETE_FEATURES

📊 Loading feature data...
✅ Loaded feature data: 232 rows
✅ Training data: 163 rows
✅ Test data: 69 rows

📋 Sample training data:
----------------------------------------------------------------------------------------------------------------------------
|"ATHLETE_ID"  |"FEATURE_TIMESTAMP"  |"TOTAL_ACTIVITIES_30D"  |"AVG_PACE_30D"      |"AVG_HEARTRATE_30D"  |"IS_FRAUDULENT"  |
----------------------------------------------------------------------------------------------------------------------------
|180           |2025-10-01 16:19:24  |43                      |17.37111745457399   |133.45116279069768   |0                |
|21            |2025-10-01 16:19:24  |40                      |18.3797119149043    |111.3075             |0                |
|82            |2025-10-01 16:19:2

In [111]:
# Save training and test data as tables for monitoring
print("💾 Saving training and test datasets to STRAVA_MODEL_REGISTRY...")

# Save training data (using fully qualified table name to avoid schema context issues)
training_data.write.mode("overwrite").save_as_table("STRAVA_DEMO_SAMPLE.STRAVA_MODEL_REGISTRY.FRAUD_DETECTION_TRAIN_DATA")
print("✅ Training data saved as FRAUD_DETECTION_TRAIN_DATA")

# Save test data (using fully qualified table name)
test_data.write.mode("overwrite").save_as_table("STRAVA_DEMO_SAMPLE.STRAVA_MODEL_REGISTRY.FRAUD_DETECTION_TEST_DATA")
print("✅ Test data saved as FRAUD_DETECTION_TEST_DATA")

# Switch to model registry schema for verification
session.use_schema("STRAVA_MODEL_REGISTRY")

# Verify the tables were created
print("\n📊 Verifying saved datasets:")
train_count = session.table("FRAUD_DETECTION_TRAIN_DATA").count()
test_count = session.table("FRAUD_DETECTION_TEST_DATA").count()
print(f"  - Training data: {train_count} rows")
print(f"  - Test data: {test_count} rows")


💾 Saving training and test datasets to STRAVA_MODEL_REGISTRY...
✅ Training data saved as FRAUD_DETECTION_TRAIN_DATA
✅ Test data saved as FRAUD_DETECTION_TEST_DATA

📊 Verifying saved datasets:
  - Training data: 163 rows
  - Test data: 69 rows


## 4. Create Inference Stored Procedure

We'll create a stored procedure that can run inference on new data using our registered models. This is essential for model monitoring as it allows us to generate predictions that can be compared against actual outcomes.


In [112]:
# Create inference stored procedure
print("🔧 Creating inference stored procedure...")

# Create a persistent ML_STAGE in the MODEL_REGISTRY schema if it doesn't exist
try:
    session.sql("CREATE STAGE IF NOT EXISTS ML_STAGE").collect()
    print("✅ Persistent ML_STAGE ready in STRAVA_MODEL_REGISTRY schema")
except Exception as e:
    print(f"⚠️ Stage creation warning: {e}")

def fraud_detection_inference_sproc(session: snowpark.Session, table_name: str, model_name: str, model_version: str) -> str:
    """
    Stored procedure to run fraud detection inference on a given table
    
    Args:
        session: Snowpark session
        table_name: Name of the table containing input data
        model_name: Name of the model in the registry
        model_version: Version of the model to use
    
    Returns:
        Success message
    """
    from snowflake.ml.registry import Registry
    from snowflake.snowpark.functions import col
    
    # Create Registry with explicit database and schema
    # Note: Cannot use USE statements inside stored procedures
    reg = Registry(
        session=session,
        database_name="STRAVA_DEMO_SAMPLE",
        schema_name="STRAVA_MODEL_REGISTRY"
    )
    
    # Get model from registry
    model = reg.get_model(model_name)
    model_version_obj = model.version(model_version)
    
    # Read input data (use fully qualified name if needed)
    input_df = session.table(table_name)
    
    # Generate predictions
    predictions = model_version_obj.run(input_df, function_name="predict")
    
    # Create prediction column name based on model version
    pred_col_name = f"{model_version.upper()}_PREDICTION"
    
    # Get just the prediction column with ATHLETE_ID for joining
    pred_only = predictions.select(
        "ATHLETE_ID",
        col('"FRAUD_PREDICTION"').alias(pred_col_name)
    )
    
    # Join predictions with original data to add the new prediction column
    result_df = input_df.join(pred_only, on="ATHLETE_ID", how="left")
    
    # Save results back to the same table (overwrite to update with new column)
    result_df.write.mode("overwrite").save_as_table(table_name)
    
    return f"Successfully generated predictions for {model_name} {model_version} on table {table_name}"

# Register the stored procedure with stage location
session.sproc.register(
    func=fraud_detection_inference_sproc,
    name="fraud_detection_inference",
    replace=True,
    is_permanent=True,
    stage_location="@ML_STAGE",
    packages=['snowflake-ml-python', 'snowflake-snowpark-python'],
    return_type=StringType()
)

print("✅ Inference stored procedure created: fraud_detection_inference")
print("💡 Registry configured with explicit database/schema for cross-session compatibility")


🔧 Creating inference stored procedure...
✅ Persistent ML_STAGE ready in STRAVA_MODEL_REGISTRY schema
✅ Inference stored procedure created: fraud_detection_inference
💡 Registry configured with explicit database/schema for cross-session compatibility


## 5. Generate Predictions for Monitoring

Run inference on both training and test datasets using both model versions. This creates the prediction data needed for model monitoring.


In [113]:
# Run inference on training data for both model versions
print("🔄 Running inference on training data...")

# Helper function to run inference (works in both Snowflake UI and locally)
def run_inference_on_table(table_name, model_version, is_training=True):
    """Run inference and add prediction column to table"""
    from snowflake.snowpark.functions import col
    
    # Try stored procedure first (works in Snowflake Notebooks)
    try:
        result = session.call("fraud_detection_inference", table_name, "fraud_detection_rf", model_version.lower())
        return result
    except Exception as sproc_error:
        # Fallback: Direct Python inference (works locally in Cursor)
        print(f"   ℹ️ Using direct inference (local mode)")
        
        # Get model
        model = mr.get_model("fraud_detection_rf")
        model_obj = model.version(model_version.upper())
        
        # Read current data and drop prediction column if it exists (to avoid duplicate column issues)
        data_df = session.table(table_name)
        pred_col_name = f"{model_version.upper()}_PREDICTION"
        
        # Drop the prediction column if it already exists
        if pred_col_name in data_df.columns:
            data_df = data_df.drop(pred_col_name)
        
        # Run prediction
        predictions = model_obj.run(data_df, function_name="predict")
        
        # Get just the prediction column with ATHLETE_ID for joining
        pred_only = predictions.select(
            "ATHLETE_ID",
            col('"FRAUD_PREDICTION"').alias(pred_col_name)
        )
        
        # Join with original data to add the new prediction column
        result_df = data_df.join(pred_only, on="ATHLETE_ID", how="left")
        
        # Save back
        result_df.write.mode("overwrite").save_as_table(table_name)
        
        return f"Successfully generated predictions for fraud_detection_rf {model_version} on table {table_name}"

# Run inference for V1 and V2 on training data
result_v1_train = run_inference_on_table("FRAUD_DETECTION_TRAIN_DATA", "v1", is_training=True)
print(f"✅ V1 Training: {result_v1_train}")

result_v2_train = run_inference_on_table("FRAUD_DETECTION_TRAIN_DATA", "v2", is_training=True)
print(f"✅ V2 Training: {result_v2_train}")


🔄 Running inference on training data...
   ℹ️ Using direct inference (local mode)
✅ V1 Training: Successfully generated predictions for fraud_detection_rf v1 on table FRAUD_DETECTION_TRAIN_DATA
   ℹ️ Using direct inference (local mode)
✅ V2 Training: Successfully generated predictions for fraud_detection_rf v2 on table FRAUD_DETECTION_TRAIN_DATA


In [114]:
# Run inference on test data for both model versions
print("🔄 Running inference on test data...")

# Run inference for V1 and V2 on test data (uses helper function from previous cell)
result_v1_test = run_inference_on_table("FRAUD_DETECTION_TEST_DATA", "v1", is_training=False)
print(f"✅ V1 Test: {result_v1_test}")

result_v2_test = run_inference_on_table("FRAUD_DETECTION_TEST_DATA", "v2", is_training=False)
print(f"✅ V2 Test: {result_v2_test}")


🔄 Running inference on test data...
   ℹ️ Using direct inference (local mode)
✅ V1 Test: Successfully generated predictions for fraud_detection_rf v1 on table FRAUD_DETECTION_TEST_DATA
   ℹ️ Using direct inference (local mode)
✅ V2 Test: Successfully generated predictions for fraud_detection_rf v2 on table FRAUD_DETECTION_TEST_DATA


In [115]:
# Verify data is ready for monitoring
print("🔍 Verifying prediction columns exist...")

# Check training data
train_cols = session.table("FRAUD_DETECTION_TRAIN_DATA").columns
print(f"\n✅ Training data columns: {', '.join(train_cols)}")

# Check test data
test_cols = session.table("FRAUD_DETECTION_TEST_DATA").columns
print(f"✅ Test data columns: {', '.join(test_cols)}")

# Verify predictions exist
if "V1_PREDICTION" in train_cols and "V2_PREDICTION" in train_cols:
    print("\n✅ All prediction columns present - ready for monitoring!")
else:
    print("\n⚠️ Missing prediction columns - run inference cells above first")
    raise Exception("Prediction columns not found")


🔍 Verifying prediction columns exist...

✅ Training data columns: ATHLETE_ID, FEATURE_TIMESTAMP, TOTAL_ACTIVITIES_30D, AVG_PACE_30D, AVG_HEARTRATE_30D, IS_FRAUDULENT, V1_PREDICTION, V2_PREDICTION
✅ Test data columns: ATHLETE_ID, FEATURE_TIMESTAMP, TOTAL_ACTIVITIES_30D, AVG_PACE_30D, AVG_HEARTRATE_30D, IS_FRAUDULENT, V1_PREDICTION, V2_PREDICTION

✅ All prediction columns present - ready for monitoring!


## 6. Create Native Snowflake Model Monitors

Now we'll create native Snowflake Model Monitors using ML Observability. These monitors will automatically track model performance, detect drift, and provide metrics visible in the Snowflake UI.


In [ ]:
# Create Model Monitor for V1 (baseline model)
print("🔧 Setting up Snowflake Model Monitor for V1...")

# Check if monitor already exists
try:
    existing_monitors = session.sql("SHOW MODEL MONITORS").collect()
    monitor_exists = any(m['name'] == 'FRAUD_DETECTION_V1_MONITOR' for m in existing_monitors)
    
    if monitor_exists:
        print("ℹ️ Model Monitor FRAUD_DETECTION_V1_MONITOR already exists")
        print("   Preserving existing monitor to keep history")
        print("   To recreate, manually drop it first: DROP MODEL MONITOR FRAUD_DETECTION_V1_MONITOR")
    else:
        create_monitor_v1_sql = """
        CREATE MODEL MONITOR FRAUD_DETECTION_V1_MONITOR
        WITH
            MODEL = FRAUD_DETECTION_RF
            VERSION = V1
            FUNCTION = predict
            SOURCE = FRAUD_DETECTION_TEST_DATA
            BASELINE = FRAUD_DETECTION_TRAIN_DATA
            TIMESTAMP_COLUMN = FEATURE_TIMESTAMP
            PREDICTION_CLASS_COLUMNS = (V1_PREDICTION)  
            ACTUAL_CLASS_COLUMNS = (IS_FRAUDULENT)
            ID_COLUMNS = (ATHLETE_ID)
            WAREHOUSE = STRAVA_DEMO_WH
            REFRESH_INTERVAL = '20 minutes'
            AGGREGATION_WINDOW = '1 day';
        """
        
        session.sql(create_monitor_v1_sql).collect()
        print("✅ Model Monitor created for V1: FRAUD_DETECTION_V1_MONITOR")
        print("   - Refresh Interval: 20 minutes")
        print("   - Aggregation Window: 1 day")
        print("   - Monitoring: Performance, Drift, and Data Quality")
        
except Exception as e:
    print(f"❌ Error setting up V1 monitor: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Ensure you have the right Snowflake edition (Enterprise or higher)")
    print("   2. Check that prediction columns exist (run inference cells first)")
    print("   3. Verify MODEL and VERSION names match registry")
    raise


🔧 Setting up Snowflake Model Monitor for V1...
ℹ️ Model Monitor FRAUD_DETECTION_V1_MONITOR already exists
   Preserving existing monitor to keep history
   To recreate, manually drop it first: DROP MODEL MONITOR FRAUD_DETECTION_V1_MONITOR


In [ ]:
# Create Model Monitor for V2 (optimized model)
print("🔧 Setting up Snowflake Model Monitor for V2...")

# Check if monitor already exists
try:
    existing_monitors = session.sql("SHOW MODEL MONITORS").collect()
    monitor_exists = any(m['name'] == 'FRAUD_DETECTION_V2_MONITOR' for m in existing_monitors)
    
    if monitor_exists:
        print("ℹ️ Model Monitor FRAUD_DETECTION_V2_MONITOR already exists")
        print("   Preserving existing monitor to keep history")
        print("   To recreate, manually drop it first: DROP MODEL MONITOR FRAUD_DETECTION_V2_MONITOR")
    else:
        create_monitor_v2_sql = """
        CREATE MODEL MONITOR FRAUD_DETECTION_V2_MONITOR
        WITH
            MODEL = FRAUD_DETECTION_RF
            VERSION = V2
            FUNCTION = predict
            SOURCE = FRAUD_DETECTION_TEST_DATA
            BASELINE = FRAUD_DETECTION_TRAIN_DATA
            TIMESTAMP_COLUMN = FEATURE_TIMESTAMP
            PREDICTION_CLASS_COLUMNS = (V2_PREDICTION)  
            ACTUAL_CLASS_COLUMNS = (IS_FRAUDULENT)
            ID_COLUMNS = (ATHLETE_ID)
            WAREHOUSE = STRAVA_DEMO_WH
            REFRESH_INTERVAL = '20 minutes'
            AGGREGATION_WINDOW = '1 day';
        """
        
        session.sql(create_monitor_v2_sql).collect()
        print("✅ Model Monitor created for V2: FRAUD_DETECTION_V2_MONITOR")
        print("   - Refresh Interval: 20 minutes")
        print("   - Aggregation Window: 1 day")
        print("   - Monitoring: Performance, Drift, and Data Quality")
        
except Exception as e:
    print(f"❌ Error setting up V2 monitor: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Ensure you have the right Snowflake edition (Enterprise or higher)")
    print("   2. Check that prediction columns exist (run inference cells first)")
    print("   3. Verify MODEL and VERSION names match registry")
    raise


🔧 Setting up Snowflake Model Monitor for V2...
ℹ️ Model Monitor FRAUD_DETECTION_V2_MONITOR already exists
   Preserving existing monitor to keep history
   To recreate, manually drop it first: DROP MODEL MONITOR FRAUD_DETECTION_V2_MONITOR


## 7. View Model Monitors

List the created monitors and check their status. The monitors will appear in the Snowflake UI under your Model Registry.


In [118]:
# List all model monitors
print("📋 Listing Model Monitors...")

try:
    monitors_sql = "SHOW MODEL MONITORS"
    monitors = session.sql(monitors_sql).collect()
    
    if monitors:
        print(f"✅ Found {len(monitors)} model monitor(s):\n")
        for monitor in monitors:
            # Snowpark Row objects use bracket notation, not .get()
            print(f"📊 Monitor: {monitor['name']}")
            # Access attributes directly with try/except for missing fields
            try:
                print(f"   - Database: {monitor['database_name']}")
            except:
                pass
            try:
                print(f"   - Schema: {monitor['schema_name']}")
            except:
                pass
            try:
                print(f"   - Created: {monitor['created_on']}")
            except:
                pass
            print()
    else:
        print("⚠️ No model monitors found")
        
except Exception as e:
    print(f"⚠️ Error listing monitors: {e}")
    print("💡 Model monitors might still be initializing")

print("\n🎯 How to View Monitors in Snowflake UI:")
print("   1. Go to Snowsight")
print("   2. Navigate to: Data > Databases > STRAVA_DEMO_SAMPLE > STRAVA_MODEL_REGISTRY")
print("   3. Click on FRAUD_DETECTION_RF model")
print("   4. You'll see the 'Monitors' tab with both V1 and V2 monitors")
print("   5. Click on each monitor to view detailed metrics and drift analysis")


📋 Listing Model Monitors...
✅ Found 2 model monitor(s):

📊 Monitor: FRAUD_DETECTION_V1_MONITOR
   - Database: STRAVA_DEMO_SAMPLE
   - Schema: STRAVA_MODEL_REGISTRY
   - Created: 2025-10-01 16:37:21.127000-07:00

📊 Monitor: FRAUD_DETECTION_V2_MONITOR
   - Database: STRAVA_DEMO_SAMPLE
   - Schema: STRAVA_MODEL_REGISTRY
   - Created: 2025-10-01 16:37:25.967000-07:00


🎯 How to View Monitors in Snowflake UI:
   1. Go to Snowsight
   2. Navigate to: Data > Databases > STRAVA_DEMO_SAMPLE > STRAVA_MODEL_REGISTRY
   3. Click on FRAUD_DETECTION_RF model
   4. You'll see the 'Monitors' tab with both V1 and V2 monitors
   5. Click on each monitor to view detailed metrics and drift analysis


## 8. Query Model Drift Metrics

Query drift metrics from the model monitors to detect data distribution changes over time.


In [119]:
# Query drift metrics for V1 monitor
print("🔍 Querying drift metrics for V1 model...")

drift_v1_sql = """
SELECT * FROM TABLE(MODEL_MONITOR_DRIFT_METRIC(
    'FRAUD_DETECTION_V1_MONITOR',
    'DIFFERENCE_OF_MEANS',
    'V1_PREDICTION',
    '1 DAY',
    DATEADD(DAY, -7, CURRENT_DATE()),
    CURRENT_DATE()
))
ORDER BY EVENT_TIMESTAMP DESC
LIMIT 10;
"""

try:
    drift_v1 = session.sql(drift_v1_sql).collect()
    if drift_v1:
        print("✅ V1 Drift Metrics (last 7 days):")
        for row in drift_v1:
            print(f"  - Timestamp: {row['EVENT_TIMESTAMP']}")
            print(f"    Column: {row['COLUMN_NAME']}")
            print(f"    Metric: {row['METRIC_NAME']}")
            print(f"    Drift Value: {row['METRIC_VALUE']:.4f}")
            print(f"    Count Used: {row['COUNT_USED']}")
            print()
    else:
        print("⚠️ No drift metrics available yet for V1")
        print("💡 Monitors need time to collect data (wait for refresh interval)")
except Exception as e:
    print(f"⚠️ Error querying V1 drift: {e}")
    print("💡 This is normal if monitors were just created - wait for first refresh")


🔍 Querying drift metrics for V1 model...
✅ V1 Drift Metrics (last 7 days):
  - Timestamp: 2025-09-30 00:00:00
    Column: V1_PREDICTION
    Metric: DIFFERENCE_OF_MEANS
    Drift Value: -0.0069
    Count Used: 1028.0

  - Timestamp: 2025-09-29 00:00:00
    Column: V1_PREDICTION
    Metric: DIFFERENCE_OF_MEANS
    Drift Value: 0.0200
    Count Used: 894.0

  - Timestamp: 2025-09-28 00:00:00
    Column: V1_PREDICTION
    Metric: DIFFERENCE_OF_MEANS
    Drift Value: -0.0130
    Count Used: 882.0

  - Timestamp: 2025-09-27 00:00:00
    Column: V1_PREDICTION
    Metric: DIFFERENCE_OF_MEANS
    Drift Value: -0.0099
    Count Used: 884.0

  - Timestamp: 2025-09-26 00:00:00
    Column: V1_PREDICTION
    Metric: DIFFERENCE_OF_MEANS
    Drift Value: -0.0002
    Count Used: 944.0

  - Timestamp: 2025-09-25 00:00:00
    Column: V1_PREDICTION
    Metric: DIFFERENCE_OF_MEANS
    Drift Value: -0.0240
    Count Used: 666.0

  - Timestamp: 2025-09-24 00:00:00
    Column: V1_PREDICTION
    Metric: DIFFER

## 9. Summary and Next Steps

Native Snowflake Model Monitors are now active and will automatically track performance, drift, and data quality.


In [120]:
# Summary
print("📋 Native Snowflake Model Monitoring - Setup Complete!")
print("="*60)

print("\n✅ What We've Accomplished:")
print("  📊 Created 2 native Snowflake Model Monitors (V1 and V2)")
print("  🔍 Set up automatic drift detection")
print("  📈 Enabled performance tracking and metrics")
print("  ⏰ Configured refresh intervals (1hr for V1, 2hr for V2)")
print("  🎯 Established baseline and source data for comparison")

print("\n🎯 What the Monitors Track:")
print("  • Model Performance Metrics (Accuracy, Precision, Recall, F1)")
print("  • Data Drift Detection (input feature distribution changes)")
print("  • Prediction Drift (output distribution changes)")
print("  • Data Quality Issues")
print("  • Model Degradation Over Time")

print("\n📊 Where to View Monitor Results:")
print("  1. Snowflake UI: Data > STRAVA_DEMO_SAMPLE > STRAVA_MODEL_REGISTRY")
print("  2. Click on FRAUD_DETECTION_RF model")
print("  3. View 'Monitors' tab for dashboards and metrics")
print("  4. Each monitor shows: Performance, Drift, Data Quality")

print("\n🔄 Monitor Refresh Schedule:")
print("  • V1 Monitor: Refreshes every 20 minutes")
print("  • V2 Monitor: Refreshes every 20 minutes")
print("  • Aggregation Window: 1 day")
print("  • Warehouse: STRAVA_DEMO_WH")

print("\n💡 Query Monitor Metrics:")
print("  Use MODEL_MONITOR_DRIFT_METRIC() function for drift")
print("  Use MODEL_MONITOR_PERFORMANCE_METRICS() for performance")
print("  See cells above for example queries")

print("\n🚀 Next Steps:")
print("  1. Wait for first refresh (~20 minutes) to see initial metrics")
print("  2. Check Snowflake UI 'Monitors' tab for visual dashboards")
print("  3. Set up alerts based on drift/performance thresholds")
print("  4. Configure notifications (email/Slack) for critical issues")
print("  5. Create automated retraining triggers based on metrics")

print("\n🎉 Native Model Monitoring is ACTIVE!")
print("✅ Your fraud detection models are now monitored with Snowflake ML Observability!")


📋 Native Snowflake Model Monitoring - Setup Complete!

✅ What We've Accomplished:
  📊 Created 2 native Snowflake Model Monitors (V1 and V2)
  🔍 Set up automatic drift detection
  📈 Enabled performance tracking and metrics
  ⏰ Configured refresh intervals (1hr for V1, 2hr for V2)
  🎯 Established baseline and source data for comparison

🎯 What the Monitors Track:
  • Model Performance Metrics (Accuracy, Precision, Recall, F1)
  • Data Drift Detection (input feature distribution changes)
  • Prediction Drift (output distribution changes)
  • Data Quality Issues
  • Model Degradation Over Time

📊 Where to View Monitor Results:
  1. Snowflake UI: Data > STRAVA_DEMO_SAMPLE > STRAVA_MODEL_REGISTRY
  2. Click on FRAUD_DETECTION_RF model
  3. View 'Monitors' tab for dashboards and metrics
  4. Each monitor shows: Performance, Drift, Data Quality

🔄 Monitor Refresh Schedule:
  • V1 Monitor: Refreshes every 20 minutes
  • V2 Monitor: Refreshes every 20 minutes
  • Aggregation Window: 1 day
  • Wa

## 10. Generate Realistic Monitoring Data with Variations

For more interesting and realistic monitoring dashboards, we'll create data with:
1. **Varied sample sizes** per day (not flat prediction counts)
2. **Gradual feature drift** over time (activity patterns changing)
3. **Natural fraud rate variations** (slight differences day-to-day)
4. **Random noise** to simulate real-world data

This creates wavy graphs instead of flat lines, demonstrating drift detection and performance tracking.


In [121]:
# Generate realistic monitoring data with variations
print("🎨 Creating realistic data with variations for monitoring...")
print("=" * 60)

from datetime import datetime, timedelta
import random

# Clear existing test data to start fresh
session.sql("TRUNCATE TABLE FRAUD_DETECTION_TEST_DATA").collect()
print("✅ Cleared existing test data for fresh start")

# Create 7 daily snapshots with increasing drift
print("\n📊 Creating 7 days of data with gradual drift and variations...")

for days_ago in range(7, 0, -1):
    timestamp = datetime.now() - timedelta(days=days_ago)
    timestamp_str = timestamp.strftime('%Y-%m-%d %H:%M:%S')
    
    # Calculate drift factor (increases over time)
    drift_factor = (7 - days_ago) / 7.0  # 0.0 to 0.86
    noise_level = drift_factor * 0.3  # Up to 30% noise
    
    # Vary sample size for each day (creates wavy prediction count graphs)
    sample_size = 50 + random.randint(-10, 10)
    
    print(f"  Day {8-days_ago}: {timestamp.strftime('%Y-%m-%d')} - {sample_size} rows, drift: {drift_factor:.2f}")
    
    # Insert data with drift - features shift over time
    insert_sql = f"""
    INSERT INTO FRAUD_DETECTION_TEST_DATA
    (ATHLETE_ID, TOTAL_ACTIVITIES_30D, AVG_PACE_30D, AVG_HEARTRATE_30D,
     IS_FRAUDULENT, FEATURE_TIMESTAMP)
    SELECT
        ATHLETE_ID,
        -- Add drift: increase activity counts over time
        CAST(TOTAL_ACTIVITIES_30D * (1 + {drift_factor * 0.2}) AS INTEGER) as TOTAL_ACTIVITIES_30D,
        -- Add noise to pace (simulates real-world variation)
        AVG_PACE_30D * (1 + UNIFORM(-{noise_level}, {noise_level}, RANDOM())) as AVG_PACE_30D,
        -- Add noise to heart rate
        AVG_HEARTRATE_30D * (1 + UNIFORM(-{noise_level}, {noise_level}, RANDOM())) as AVG_HEARTRATE_30D,
        IS_FRAUDULENT,
        '{timestamp_str}'::TIMESTAMP_NTZ as FEATURE_TIMESTAMP
    FROM (
        SELECT *
        FROM STRAVA_DEMO_SAMPLE.STRAVA_FEATURE_STORE.ATHLETE_FEATURES
        ORDER BY RANDOM()
        LIMIT {sample_size}
    )
    """
    
    session.sql(insert_sql).collect()

total = session.table("FRAUD_DETECTION_TEST_DATA").count()
print(f"\n✅ Created {total} total rows across 7 days with variations")
print("\n💡 Next step: Run inference to populate prediction columns")


🎨 Creating realistic data with variations for monitoring...
✅ Cleared existing test data for fresh start

📊 Creating 7 days of data with gradual drift and variations...
  Day 1: 2025-09-24 - 58 rows, drift: 0.00
  Day 2: 2025-09-25 - 58 rows, drift: 0.14
  Day 3: 2025-09-26 - 47 rows, drift: 0.29
  Day 4: 2025-09-27 - 58 rows, drift: 0.43
  Day 5: 2025-09-28 - 59 rows, drift: 0.57
  Day 6: 2025-09-29 - 40 rows, drift: 0.71
  Day 7: 2025-09-30 - 50 rows, drift: 0.86

✅ Created 370 total rows across 7 days with variations

💡 Next step: Run inference to populate prediction columns


In [ ]:
# Run inference on all the time-series data
print("🔄 Running inference on time-series data...")
print("=" * 60)

# First, drop prediction columns if they exist (to avoid collision)
print("\n🔧 Preparing table for inference...")
try:
    session.sql("ALTER TABLE FRAUD_DETECTION_TEST_DATA DROP COLUMN IF EXISTS V1_PREDICTION").collect()
    session.sql("ALTER TABLE FRAUD_DETECTION_TEST_DATA DROP COLUMN IF EXISTS V2_PREDICTION").collect()
    print("✅ Removed existing prediction columns")
except Exception as e:
    print(f"ℹ️  Prediction columns already removed or don't exist: {e}")

# Run inference for both model versions (uses helper function from cell 12)
print("\n🤖 Generating V1 predictions...")
result_v1 = run_inference_on_table("FRAUD_DETECTION_TEST_DATA", "v1", is_training=False)
print(f"✅ {result_v1}")

print("\n🤖 Generating V2 predictions...")
result_v2 = run_inference_on_table("FRAUD_DETECTION_TEST_DATA", "v2", is_training=False)
print(f"✅ {result_v2}")

# Verify the data now has predictions and timestamps
print("\n📊 Verifying time-series data...")

# Check final columns
final_cols = session.table("FRAUD_DETECTION_TEST_DATA").columns
has_v1 = "V1_PREDICTION" in final_cols
has_v2 = "V2_PREDICTION" in final_cols

if has_v1 and has_v2:
    sample_data = session.sql("""
        SELECT 
            FEATURE_TIMESTAMP,
            COUNT(*) as row_count,
            AVG(V1_PREDICTION) as avg_v1_pred,
            AVG(V2_PREDICTION) as avg_v2_pred
        FROM FRAUD_DETECTION_TEST_DATA
        GROUP BY FEATURE_TIMESTAMP
        ORDER BY FEATURE_TIMESTAMP DESC
        LIMIT 10
    """).collect()
    
    print("\n📅 Time-series snapshot summary:")
    for row in sample_data:
        print(f"  {row['FEATURE_TIMESTAMP']}: {row['ROW_COUNT']} rows, "
              f"V1 avg: {row['AVG_V1_PRED']:.3f}, V2 avg: {row['AVG_V2_PRED']:.3f}")
    
    print("\n✅ Time-series data ready for monitoring!")
    print("💡 The model monitors will now have multiple data points over time")
else:
    print(f"\n⚠️ Warning: Missing prediction columns!")
    print(f"   V1_PREDICTION exists: {has_v1}")
    print(f"   V2_PREDICTION exists: {has_v2}")
    print(f"   Final columns: {', '.join(final_cols)}")
    print("\n💡 Check the cell output above for errors")


🔄 Running inference on time-series data...

🔧 Preparing table for inference...
✅ Removed existing prediction columns

🤖 Generating V1 predictions...
   ℹ️ Using direct inference (local mode)
✅ Successfully generated predictions for fraud_detection_rf v1 on table FRAUD_DETECTION_TEST_DATA

🤖 Generating V2 predictions...
   ℹ️ Using direct inference (local mode)
✅ Successfully generated predictions for fraud_detection_rf v2 on table FRAUD_DETECTION_TEST_DATA

🎯 Simulating V2 improved performance...
✅ V2 now correctly predicts ~30% of V1's errors (demonstrating improvement)

📊 Verifying time-series data...

📅 Time-series snapshot summary:
  2025-09-30 16:44:28: 932 rows, V1 avg: 0.089, V2 avg: 0.089
  2025-09-29 16:44:27: 815 rows, V1 avg: 0.124, V2 avg: 0.125
  2025-09-28 16:44:27: 901 rows, V1 avg: 0.079, V2 avg: 0.074
  2025-09-27 16:44:26: 973 rows, V1 avg: 0.081, V2 avg: 0.082
  2025-09-26 16:44:26: 1225 rows, V1 avg: 0.033, V2 avg: 0.034
  2025-09-25 16:44:25: 1068 rows, V1 avg: 0.0

## 11. Force Monitor Refresh (Optional)

**Optional:** If you want to see the latest monitoring graphs immediately instead of waiting ~20 minutes, you can drop and recreate the monitors. This forces them to process all data right away.

⚠️ **Note:** This will reset monitor history. Only run this if you want immediate results for demo purposes.


In [123]:
# Force monitor refresh by recreating them
print("🔄 Forcing monitor refresh by dropping and recreating...")
print("=" * 60)
print("⚠️  This will reset monitor history but give you immediate results")
print()

# Drop existing monitors
print("🗑️  Dropping existing monitors...")
try:
    session.sql("DROP MODEL MONITOR IF EXISTS FRAUD_DETECTION_V1_MONITOR").collect()
    print("   ✅ Dropped V1 monitor")
except Exception as e:
    print(f"   ⚠️  V1 drop: {e}")

try:
    session.sql("DROP MODEL MONITOR IF EXISTS FRAUD_DETECTION_V2_MONITOR").collect()
    print("   ✅ Dropped V2 monitor")
except Exception as e:
    print(f"   ⚠️  V2 drop: {e}")

# Recreate V1 monitor with fresh data
print("\n✅ Creating V1 monitor...")
create_v1_sql = """
CREATE MODEL MONITOR FRAUD_DETECTION_V1_MONITOR
WITH
    MODEL = FRAUD_DETECTION_RF
    VERSION = V1
    FUNCTION = predict
    SOURCE = FRAUD_DETECTION_TEST_DATA
    BASELINE = FRAUD_DETECTION_TRAIN_DATA
    TIMESTAMP_COLUMN = FEATURE_TIMESTAMP
    PREDICTION_CLASS_COLUMNS = (V1_PREDICTION)  
    ACTUAL_CLASS_COLUMNS = (IS_FRAUDULENT)
    ID_COLUMNS = (ATHLETE_ID)
    WAREHOUSE = STRAVA_DEMO_WH
    REFRESH_INTERVAL = '20 minutes'
    AGGREGATION_WINDOW = '1 day';
"""
session.sql(create_v1_sql).collect()
print("   ✅ V1 monitor created")

# Recreate V2 monitor
print("\n✅ Creating V2 monitor...")
create_v2_sql = """
CREATE MODEL MONITOR FRAUD_DETECTION_V2_MONITOR
WITH
    MODEL = FRAUD_DETECTION_RF
    VERSION = V2
    FUNCTION = predict
    SOURCE = FRAUD_DETECTION_TEST_DATA
    BASELINE = FRAUD_DETECTION_TRAIN_DATA
    TIMESTAMP_COLUMN = FEATURE_TIMESTAMP
    PREDICTION_CLASS_COLUMNS = (V2_PREDICTION)  
    ACTUAL_CLASS_COLUMNS = (IS_FRAUDULENT)
    ID_COLUMNS = (ATHLETE_ID)
    WAREHOUSE = STRAVA_DEMO_WH
    REFRESH_INTERVAL = '20 minutes'
    AGGREGATION_WINDOW = '1 day';
"""
session.sql(create_v2_sql).collect()
print("   ✅ V2 monitor created")

# Verify
print("\n🔍 Verifying monitors...")
monitors = session.sql("SHOW MODEL MONITORS").collect()
for m in monitors:
    baseline_info = eval(m['baseline'])
    print(f"\n📊 {m['name']}:")
    print(f"   Baseline: {baseline_info['name']} (Status: {baseline_info['status']})")
    print(f"   Monitor state: {m['monitor_state']}")
    print(f"   Refresh interval: {m['refresh_interval']}")

print("\n🎉 Monitors recreated with fresh data!")
print("💡 Check Snowflake UI Monitoring tab now - graphs should show variations")
print("💡 Navigate to: Data > STRAVA_DEMO_SAMPLE > STRAVA_MODEL_REGISTRY > FRAUD_DETECTION_RF > Monitoring")


🔄 Forcing monitor refresh by dropping and recreating...
⚠️  This will reset monitor history but give you immediate results

🗑️  Dropping existing monitors...
   ✅ Dropped V1 monitor
   ✅ Dropped V2 monitor

✅ Creating V1 monitor...
   ✅ V1 monitor created

✅ Creating V2 monitor...
   ✅ V2 monitor created

🔍 Verifying monitors...

📊 FRAUD_DETECTION_V1_MONITOR:
   Baseline: FRAUD_DETECTION_TRAIN_DATA (Status: ACTIVE)
   Monitor state: ACTIVE
   Refresh interval: 20 minutes

📊 FRAUD_DETECTION_V2_MONITOR:
   Baseline: FRAUD_DETECTION_TRAIN_DATA (Status: ACTIVE)
   Monitor state: ACTIVE
   Refresh interval: 20 minutes

🎉 Monitors recreated with fresh data!
💡 Check Snowflake UI Monitoring tab now - graphs should show variations
💡 Navigate to: Data > STRAVA_DEMO_SAMPLE > STRAVA_MODEL_REGISTRY > FRAUD_DETECTION_RF > Monitoring
